# 04 · Robustez de las decisiones metodológicas

**Proyecto final · Reactividad alfa en EEG durante la alternancia de ojos abiertos y cerrados**

| | |
|---|---|
| Dueño | D · Ascencio |
| Etapas de la rúbrica | Etapas 5 y 6 (20 % + 15 % + 10 %) |
| Entregable técnico | Al menos 2 variantes de `CONFIG` y tabla de robustez |
| Figuras | fig06_robustez (razón EC/EO bajo cada variante) |

Reglas: este notebook solo lo edita su dueño. Se abre desde GitHub, se guarda con *Archivo → Guardar una copia en GitHub* en tu rama
(nunca en `main`) y antes de guardar se borran las salidas (*Editar → Borrar todas las salidas*). Las figuras se guardan con
`alfa.guardar_figura(fig, "figNN_nombre")` en la carpeta `figuras/` de Drive.

## 0. Preparación

In [ ]:
# ── Preparación (igual en todos los notebooks; no editar) ──────────────
RAMA = "main"   # cámbiala solo si necesitas un alfa.py que aún está en un PR

import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/repo"
    URL = "https://github.com/moiseiseiseis/Alpha-reactivity-during-resting-state-EEG.git"
    if not os.path.isdir(REPO):
        !git clone -q -b {RAMA} {URL} {REPO}
    else:
        !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} reset -q --hard origin/{RAMA}
    %pip install -q mne
    sys.path.insert(0, f"{REPO}/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

import importlib
import alfa
importlib.reload(alfa)   # recoge cambios de alfa.py sin reiniciar el entorno

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

R = alfa.rutas()
print("Datos en:", R["datos"])
print("Registro encontrado:", R["vhdr"].exists())

In [ ]:
# Parámetros en uso: todo sale de CONFIG (no escribir números a mano en el notebook)
cfg = alfa.CONFIG
alfa.imprimir_config(cfg)

In [ ]:
raw = alfa.cargar_raw(R["vhdr"])
bloques = alfa.tabla_bloques(raw)

## 1. Variantes a evaluar

- [ ] Elegir al menos 2 decisiones y justificar por qué importan
- [ ] Ventana de 2 / 5 / 10 s
- [ ] Con y sin exclusión de artefactos
- [ ] (opcional) Otros canales, otro filtro, otro recorte de transición

In [ ]:
variantes = {
    "principal": alfa.CONFIG,
    "ventana_5s": alfa.variante(ventana_s=5.0),
    "ventana_10s": alfa.variante(ventana_s=10.0),
    "sin_exclusion": alfa.variante(excluir_artefactos=False),
}

## 2. Correr el pipeline con cada variante

- [ ] Mismo código, solo cambia `cfg`

In [ ]:
filas = []
for nombre, cfg_v in variantes.items():
    raw_v = raw.copy()
    if cfg_v["excluir_artefactos"]:
        raw_v = alfa.marcar_artefactos(raw_v, cfg_v)
    raw_v = alfa.acondicionar(raw_v, cfg_v)
    res = alfa.resumen_por_bloque(alfa.potencia_alfa(raw_v, bloques, cfg_v))
    res["variante"] = nombre
    filas.append(res)
robustez = pd.concat(filas, ignore_index=True)
robustez.to_csv(R["resultados"] / "robustez.csv", index=False)

## 3. Tabla y figura de robustez

- [ ] ¿La diferencia EO/EC se mantiene, disminuye o cambia?

In [ ]:
# alfa.guardar_figura(fig, "fig06_robustez")

## 4. Fundamento fisiológico (para el informe)

- [ ] Generadores del alfa posterior, tálamo-corteza, efecto Berger
- [ ] Distribución espacial → insumo para la selección de canales de A

In [ ]:
# TODO

## Interpretación breve

> _..._